# Pipeline de datos: Exploración de las tablas principales generadas.
**Crimen y Rendimiento Escolar en Santiago · 2013–2019**

Este notebook recorre las tablas principales que produce el pipeline, 
mostrando la estructura, dimensiones y estadísticas descriptivas de cada una.

Está pensado como punto de entrada para dos casos:

- **Investigador que quiera replicar el trabajo:** puede correr el notebook 
end-to-end después de ejecutar los 8 scripts y verificar que sus outputs coinciden 
con los aquí renderizados.
- **Lector general:** puede revisar los outputs ya renderizados para entender 
la estructura de los datos sin necesidad de correr código.

---

### Tablas cubiertas

| # | Tabla | Script |
|---|-------|--------|
| 1 | `crime_severity_lookup.csv` | 01–02 |
| 2 | `crime_index_geocode_daily.csv` | 03 |
| 3 | `schools_clean_2013_2019_RM.parquet` | 04 |
| 4 | `school_locations_RM.parquet` | 04 |
| 5 | `data_education_2013_2019_RM.parquet` | 05 |
| 6 | `crime_exposure_400m.parquet` | 06 |
| 7 | `tabla_final_escuelas_crimen.parquet` | 07 |


## Setup

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

# Establece PROJECT_ROOT tanto si se corre desde notebooks/ como desde la raíz
PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd().resolve()
)
SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from project_paths import (
    CRIME_SEVERITY_LOOKUP_PATH,
    CRIME_INDEX_DAILY_PATH,
    SCHOOLS_OUTPUT_PATH,
    SCHOOL_LOCATIONS_PATH,
    EDUCATION_OUTPUT_PATH,
    CRIME_BUFFER_400M_PATH,
    FINAL_TABLE_PATH,
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Todos los paths importados correctamente.')


PROJECT_ROOT: /Users/cami/Desktop/UAI-crimen-performance/crime_education_project
Todos los paths importados correctamente.


In [2]:
def info(df, nombre):
    sep = '-' * 60
    print(sep)
    print(' ', nombre)
    print(sep)
    print('  Filas x Columnas :', df.shape[0], 'x', df.shape[1])
    print('  Columnas         :', list(df.columns))
    mem = df.memory_usage(deep=True).sum() / 1024**2
    print('  Memoria          : {:.1f} MB'.format(mem))
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print('  Nulos:')
        for col, n in nulls.items():
            print('    {}: {:,} ({:.1f}%)'.format(col, n, 100*n/len(df)))
    else:
        print('  Nulos            : ninguno')
    print()


---
## 1 · `crime_severity_lookup.csv`
**Scripts 01–02** — `config/crime_severity_lookup.csv`

Tabla de referencia con el ranking de severidad para los 98 tipos de delito. 
Es el único archivo del pipeline editado manualmente: el investigador asigna 
`rank_gravedad` (1 = más grave) y el Script 02 calcula `score_gravedad` 
automáticamente en escala [0, 1].

**8 niveles de gravedad.** Los delitos de droga están en el Nivel 5 — 
separados deliberadamente de violencia física (Niveles 1–3), lo que afecta 
la composición del Índice Cluster (v3).

In [3]:
lookup = pd.read_csv(CRIME_SEVERITY_LOOKUP_PATH)
info(lookup, 'crime_severity_lookup.csv')
lookup.head(5)


------------------------------------------------------------
  crime_severity_lookup.csv
------------------------------------------------------------
  Filas x Columnas : 98 x 6
  Columnas         : ['delito', 'rank_gravedad', 'score_gravedad', 'nivel_gravedad', 'grupo_gravedad', 'comentario']
  Memoria          : 0.0 MB
  Nulos:
    comentario: 98 (100.0%)



,delito,rank_gravedad,score_gravedad,nivel_gravedad,grupo_gravedad,comentario
0,Homicidio,1,1.0000,1,violencia_letal_extrema,NaN
1,Secuestro,2,0.9897,1,violencia_letal_extrema,NaN
2,Violaciones,3,0.9794,1,violencia_letal_extrema,NaN
3,Trata de personas,4,0.9691,1,violencia_letal_extrema,NaN
4,Tráfico de armas,5,0.9588,1,violencia_letal_extrema,NaN


In [4]:
print('Delitos por nivel de gravedad:')
print(
    lookup.groupby(['nivel_gravedad', 'grupo_gravedad'])
    .agg(n_delitos=('delito', 'count'),
         score_min=('score_gravedad', 'min'),
         score_max=('score_gravedad', 'max'))
    .to_string()
)


Delitos por nivel de gravedad:
                                            n_delitos  score_min  score_max
nivel_gravedad grupo_gravedad                                              
1              violencia_letal_extrema              5     0.9588     1.0000
2              violencia_grave                      7     0.8866     0.9485
3              armas_y_robos                       14     0.7423     0.8763
4              propiedad_lesiones_sexuales         20     0.5361     0.7320
5              drogas                               6     0.4742     0.5258
6              economicos_institucionales          13     0.3402     0.4639
7              orden_publico_infracciones          16     0.1753     0.3299
8              incivilidades                       17     0.0000     0.1649


In [5]:
print('5 delitos MAS graves:')
print(lookup.nsmallest(5, 'rank_gravedad')[['delito','rank_gravedad','score_gravedad','grupo_gravedad']].to_string(index=False))
print()
print('5 delitos MENOS graves:')
print(lookup.nlargest(5, 'rank_gravedad')[['delito','rank_gravedad','score_gravedad','grupo_gravedad']].to_string(index=False))


5 delitos MAS graves:
           delito  rank_gravedad  score_gravedad          grupo_gravedad
        Homicidio              1          1.0000 violencia_letal_extrema
        Secuestro              2          0.9897 violencia_letal_extrema
      Violaciones              3          0.9794 violencia_letal_extrema
Trata de personas              4          0.9691 violencia_letal_extrema
 Tráfico de armas              5          0.9588 violencia_letal_extrema

5 delitos MENOS graves:
                                              delito  rank_gravedad  score_gravedad grupo_gravedad
                                         Extranjería             98          0.0000  incivilidades
Ofensas al pudor o ultraje público buenas costumbres             97          0.0103  incivilidades
                              Pensiones alimenticias             96          0.0206  incivilidades
                 Especies abandonadas en vía pública             95          0.0309  incivilidades
                    

---
## 2 · `crime_index_geocode_daily.csv`
**Script 03** — `data/intermediate/crime_index_geocode_daily.csv`

Índice de delictualidad al nivel GEOCODIGO (manzana censal, 11 dígitos) × día. 
Conecta el dataset policial crudo con el cruce espacial del Script 06.

> **Fórmula:** `indice_delictualidad = sum(frecuencia * score_gravedad) / sum(frecuencia)`  
> Rango: \[0, 1\]
- 0 si no hay delitos 
- valores típicos: \[0.3–0.7\]

In [6]:
crime_daily = pd.read_csv(CRIME_INDEX_DAILY_PATH, parse_dates=['dia'])
info(crime_daily, 'crime_index_geocode_daily.csv')
crime_daily.head(5)


------------------------------------------------------------
  crime_index_geocode_daily.csv
------------------------------------------------------------
  Filas x Columnas : 2316032 x 14
  Columnas         : ['GEOCODIGO', 'dia', 'anio', 'indice_delictualidad', 'frecuencia_total', 'n_delitos_distintos', 'n_nivel_1', 'n_nivel_2', 'n_nivel_3', 'n_nivel_4', 'n_nivel_5', 'n_nivel_6', 'n_nivel_7', 'n_nivel_8']
  Memoria          : 247.4 MB
  Nulos            : ninguno



,GEOCODIGO,dia,anio,indice_delictualidad,frecuencia_total,n_delitos_distintos,n_nivel_1,n_nivel_2,n_nivel_3,n_nivel_4,n_nivel_5,n_nivel_6,n_nivel_7,n_nivel_8
0,13101011001,2013-01-02,2013,0.0808,6,2,0,0,0,0,0,0,1,5
1,13101011002,2013-01-02,2013,0.1031,1,1,0,0,0,0,0,0,0,1
2,13101011003,2013-01-02,2013,0.3952,6,4,0,0,1,2,0,0,0,3
3,13101011004,2013-01-02,2013,0.6701,4,2,0,0,0,3,1,0,0,0
4,13101011005,2013-01-02,2013,0.7732,2,1,0,0,2,0,0,0,0,0


In [7]:
print('Periodo cubierto:')
print('  Desde :', crime_daily['dia'].min().date())
print('  Hasta :', crime_daily['dia'].max().date())
print('  Dias unicos       :', crime_daily['dia'].nunique())
print('  GEOCODIGOs unicos :', crime_daily['GEOCODIGO'].nunique())
print()
print('Estadisticas del indice de delictualidad:')
print(crime_daily['indice_delictualidad'].describe().to_string())


Periodo cubierto:
  Desde : 2013-01-02
  Hasta : 2022-09-30
  Dias unicos       : 3553
  GEOCODIGOs unicos : 1457

Estadisticas del indice de delictualidad:
count   2316032.0000
mean          0.5455
std           0.2718
min           0.0000
25%           0.3196
50%           0.5567
75%           0.7732
max           1.0000


---
## 3 · `schools_clean_2013_2019_RM.parquet`
**Script 04** — `data/processed/schools_clean_2013_2019_RM.parquet`

Establecimientos educacionales de la Región Metropolitana, años: 2013–2019. 

Una fila por RBD × año. Las coordenadas corresponden a la **coordenada madre**: 

- moda de lat/lon redondeada a 4 decimales (~11m), robusta ante errores de captura.

**Nota 2013:** el directorio original no trae lat/lon — se imputan las coordenadas 
del mismo RBD en 2014.

In [8]:
schools = pd.read_parquet(SCHOOLS_OUTPUT_PATH)
info(schools, 'schools_clean_2013_2019_RM.parquet')
schools.head(3)


------------------------------------------------------------
  schools_clean_2013_2019_RM.parquet
------------------------------------------------------------
  Filas x Columnas : 26941 x 24
  Columnas         : ['rbd', 'agno', 'cod_reg_rbd', 'cod_pro_rbd', 'cod_com_rbd', 'nom_com_rbd', 'cod_depe', 'cod_depe2', 'rural_rbd', 'pago_matricula', 'pago_mensual', 'ori_religiosa', 'estado_estab', 'ens_01', 'ens_02', 'ens_03', 'ens_04', 'ens_05', 'ens_06', 'ens_07', 'ens_08', 'ens_09', 'latitud', 'longitud']
  Memoria          : 10.7 MB
  Nulos:
    cod_depe2: 3,079 (11.4%)
    estado_estab: 6,168 (22.9%)
    latitud: 134 (0.5%)
    longitud: 134 (0.5%)



,rbd,agno,cod_reg_rbd,cod_pro_rbd,cod_com_rbd,nom_com_rbd,cod_depe,cod_depe2,rural_rbd,pago_matricula,pago_mensual,ori_religiosa,estado_estab,ens_01,ens_02,ens_03,ens_04,ens_05,ens_06,ens_07,ens_08,ens_09,latitud,longitud
0,8485,2013,13,131,13101,SANTIAGO,2,NaN,0,Gratuito,Gratuito,1,None,110,310,0,0,0,0,0,0,0,-33.4449,-70.6503
1,8487,2013,13,131,13101,SANTIAGO,2,NaN,0,$1.000 a $10.000,Gratuito,1,None,110,310,0,0,0,0,0,0,0,-33.4392,-70.6573
2,8488,2013,13,131,13101,SANTIAGO,2,NaN,0,$1.000 a $10.000,Gratuito,1,None,310,0,0,0,0,0,0,0,0,-33.4456,-70.6793


In [9]:
print('RBDs unicos por año:')
print(schools.groupby('agno')['rbd'].nunique().to_string())
print()
print('Cobertura de coordenadas por año:')
cov = schools.groupby('agno').apply(
    lambda d: pd.Series({
        'con_coords': int(d['latitud'].notna().sum()),
        'total': len(d),
        'pct': '{:.1f}%'.format(100*d['latitud'].notna().mean())
    })
)
print(cov.to_string())


RBDs unicos por año:
agno
2013    3079
2014    3089
2015    4107
2016    4133
2017    4143
2018    4149
2019    4241

Cobertura de coordenadas por año:
      con_coords  total     pct
agno                           
2013        3079   3079  100.0%
2014        3089   3089  100.0%
2015        4107   4107  100.0%
2016        4133   4133  100.0%
2017        4133   4143   99.8%
2018        4133   4149   99.6%
2019        4133   4241   97.5%


In [10]:
print('Distribucion por tipo de dependencia (cod_depe2):')
print('  1=Municipal  2=Part.Subvencionado  3=Part.Pagado')
print(schools.groupby('agno')['cod_depe2'].value_counts().unstack(fill_value=0).to_string())


Distribucion por tipo de dependencia (cod_depe2):
  1=Municipal  2=Part.Subvencionado  3=Part.Pagado
cod_depe2  1.0000  2.0000  3.0000  4.0000  5.0000
agno                                             
2014          722    2046     288      33       0
2015          863    2540     671      33       0
2016          868    2539     693      33       0
2017          868    2541     701      33       0
2018          814    2511     737      33      54
2019          814    2510     830      33      54


---
## 4 · `school_locations_RM.parquet`
**Script 04** — `data/processed/school_locations_RM.parquet`

Una fila por RBD con su coordenada madre canónica. 

- Es el input geográfico que usa el Script 06 para calcular los buffers espaciales.

In [11]:
locations = pd.read_parquet(SCHOOL_LOCATIONS_PATH)
info(locations, 'school_locations_RM.parquet')
locations.head(5)


------------------------------------------------------------
  school_locations_RM.parquet
------------------------------------------------------------
  Filas x Columnas : 4133 x 7
  Columnas         : ['rbd', 'latitud', 'longitud', 'cod_reg_rbd', 'cod_com_rbd', 'nom_com_rbd', 'cod_depe2']
  Memoria          : 0.5 MB
  Nulos:
    cod_depe2: 3,079 (74.5%)



,rbd,latitud,longitud,cod_reg_rbd,cod_com_rbd,nom_com_rbd,cod_depe2
0,1646,0.0000,0.0000,13,13126,QUINTA NORMAL,3.0000
1,6880,0.0000,0.0000,13,13115,LO BARNECHEA,3.0000
2,8485,-33.4449,-70.6503,13,13101,SANTIAGO,NaN
3,8487,-33.4392,-70.6573,13,13101,SANTIAGO,NaN
4,8488,-33.4456,-70.6793,13,13101,SANTIAGO,NaN


In [12]:
print('Rango de coordenadas (Region Metropolitana):')
print('  Latitud  : [{:.4f}, {:.4f}]'.format(locations['latitud'].min(), locations['latitud'].max()))
print('  Longitud : [{:.4f}, {:.4f}]'.format(locations['longitud'].min(), locations['longitud'].max()))
print('  RBDs unicos:', locations['rbd'].nunique())


Rango de coordenadas (Region Metropolitana):
  Latitud  : [-34.0335, 0.0000]
  Longitud : [-71.5954, 0.0000]
  RBDs unicos: 4133


---
## 5 · `data_education_2013_2019_RM.parquet`
**Script 05** — `data/processed/data_education_2013_2019_RM.parquet`

Tabla de resultados SIMCE al nivel RBD × año × grado (curso), filtrada para la RM. 

- Incluye puntajes promedio de Lenguaje y Matemáticas, controles socioeconómicos 
y la fecha exacta de rendición de cada examen.

**Nota sobre corrección aplicada:** `cod_depe2` y `cod_grupo` usan esquemas 
de enteros distintos en el archivo Stata original para 2013–2017 vs 2018–2019. 
El Script 05 normaliza ambas variables a una escala única antes de guardar.

In [13]:
edu = pd.read_parquet(EDUCATION_OUTPUT_PATH)
info(edu, 'data_education_2013_2019_RM.parquet')
edu.head(3)


------------------------------------------------------------
  data_education_2013_2019_RM.parquet
------------------------------------------------------------
  Filas x Columnas : 32519 x 59
  Columnas         : ['rbd', 'agno', 'grado', 'codigo_bbdd', 'nom_rbd', 'dvrbd', 'cod_reg_rbd', 'nom_reg_rbd', 'cod_pro_rbd', 'nom_pro_rbd', 'cod_com_rbd', 'nom_com_rbd', 'cod_depe2', 'fecha_rendicion', 'id_rbd_grado', 'cod_grupo', 'nalu_2b_rbd', 'nalu_lect2b_rbd', 'prom_lect2b_rbd', 'nalu_2m_rbd', 'nalu_lect2m_rbd', 'nalu_mate2m_rbd', 'prom_lect2m_rbd', 'prom_mate2m_rbd', 'nalu_4b_rbd', 'nalu_lect4b_rbd', 'nalu_mate4b_rbd', 'nalu_nat4b_rbd', 'prom_lect4b_rbd', 'prom_mate4b_rbd', 'prom_nat4b_rbd', 'nalu_6b_rbd', 'nalu_lect6b_rbd', 'nalu_mate6b_rbd', 'prom_lect6b_rbd', 'prom_mate6b_rbd', 'nalu_8b_rbd', 'nalu_lect8b_rbd', 'nalu_mate8b_rbd', 'nalu_nat8b_rbd', 'prom_lect8b_rbd', 'prom_mate8b_rbd', 'prom_nat8b_rbd', 'nalu_nat2m_rbd', 'prom_nat2m_rbd', 'nalu_soc6b_rbd', 'prom_soc6b_rbd', 'palu_eda_ade_m

,rbd,agno,grado,codigo_bbdd,nom_rbd,dvrbd,cod_reg_rbd,nom_reg_rbd,cod_pro_rbd,nom_pro_rbd,cod_com_rbd,nom_com_rbd,cod_depe2,fecha_rendicion,id_rbd_grado,cod_grupo,nalu_2b_rbd,nalu_lect2b_rbd,prom_lect2b_rbd,nalu_2m_rbd,...,nalu_nat8b_rbd,prom_lect8b_rbd,prom_mate8b_rbd,prom_nat8b_rbd,nalu_nat2m_rbd,prom_nat2m_rbd,nalu_soc6b_rbd,prom_soc6b_rbd,palu_eda_ade_mate6b_rbd,nalu_soc2m_rbd,prom_soc2m_rbd,difgru_lect2m_rbd,difgru_mate2m_rbd,sigdif_lect2m_rbd,nalu_nat6b_rbd,prom_nat6b_rbd,nalu_soc8b_rbd,prom_soc8b_rbd,latitud,longitud
0,8513,2013,2b,final20141111v1,ESC. BAS. REPUBLICA ORIENTAL DE URUGUAY,8.0000,13.0000,REGION METROPOLITANA DE SANTIAGO,1.0000,SANTIAGO,13101.0000,SANTIAGO,1.0000,2013-10-29,8513_2b,3.0000,49.0000,49.0000,247.0000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-33.4499,-70.6403
1,8514,2013,2b,final20141111v1,LICEO ROSA ESTER ALESANDRI RODRIGUEZ,6.0000,13.0000,REGION METROPOLITANA DE SANTIAGO,1.0000,SANTIAGO,13108.0000,INDEPENDENCIA,1.0000,2013-10-29,8514_2b,3.0000,57.0000,57.0000,265.0000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-33.4098,-70.6625
2,8515,2013,2b,final20141111v1,ESCUELA BASICA CORNELIA OLIVARES,4.0000,13.0000,REGION METROPOLITANA DE SANTIAGO,1.0000,SANTIAGO,13108.0000,INDEPENDENCIA,1.0000,2013-10-29,8515_2b,3.0000,27.0000,27.0000,228.0000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-33.4231,-70.6568


In [14]:
print('Disponibilidad SIMCE por año x grado (N escuelas):')
pivot = edu.groupby(['agno', 'grado'])['rbd'].nunique().unstack(fill_value=0)
print(pivot.to_string())


Disponibilidad SIMCE por año x grado (N escuelas):
grado    2b   2m    4b    6b    8b
agno                              
2013   1598  744  1595  1563  1503
2014   1614  755  1618  1586  1520
2015   1618  767  1618  1595  1522
2016      0  772   856     0     0
2017      0  794  1744     0  1641
2018      0  777  1609  1594     0
2019      0    0     0     0  1516


In [15]:
print('Fechas exactas de rendicion por año x grado:')
fechas = edu.groupby(['agno', 'grado'])['fecha_rendicion'].first().unstack()
print(fechas.to_string())


Fechas exactas de rendicion por año x grado:
grado         2b         2m         4b         6b         8b
agno                                                        
2013  2013-10-29 2013-11-20 2013-10-09 2013-10-23 2013-11-06
2014  2014-10-23 2014-11-19 2014-10-15 2014-10-22 2014-11-12
2015  2015-10-08 2015-10-28 2015-10-07 2015-10-15 2015-10-21
2016         NaT 2016-10-19 2016-10-05        NaT        NaT
2017         NaT 2017-10-18 2017-11-08        NaT 2017-10-12
2018         NaT 2018-10-18 2018-11-07 2018-10-24        NaT
2019         NaT        NaT        NaT        NaT 2019-10-09


In [16]:
print('Puntajes SIMCE — estadisticas descriptivas:')
score_cols = [c for c in edu.columns if c.startswith('prom_')]
print(edu[score_cols].describe().round(2).to_string())


Puntajes SIMCE — estadisticas descriptivas:
       prom_lect2b_rbd  prom_lect2m_rbd  prom_mate2m_rbd  prom_lect4b_rbd  prom_mate4b_rbd  prom_nat4b_rbd  prom_lect6b_rbd  prom_mate6b_rbd  prom_lect8b_rbd  prom_mate8b_rbd  prom_nat8b_rbd  prom_nat2m_rbd  prom_soc6b_rbd  prom_soc2m_rbd  prom_nat6b_rbd  prom_soc8b_rbd
count        4810.0000        4604.0000        4603.0000        9020.0000        9018.0000       1550.0000        6321.0000        6326.0000        7683.0000        7685.0000       3139.0000       1546.0000          0.0000        793.0000       1559.0000       1511.0000
mean          249.5500         258.0500         280.1200         261.3100         254.0500        252.0900         242.7800         248.6400         239.3600         259.0100        261.4800        253.7800             NaN        261.0700        249.9800        247.5700
std            25.4600          30.1200          45.6700          26.6800          28.9800         25.9400          27.9000          31.8700   

In [17]:
print('Distribucion del grupo socioeconomico (cod_grupo normalizado):')
print('  1=Bajo  2=Medio bajo  3=Medio  4=Medio alto  5=Alto')
print(edu['cod_grupo'].value_counts(dropna=False).sort_index().to_string())


Distribucion del grupo socioeconomico (cod_grupo normalizado):
  1=Bajo  2=Medio bajo  3=Medio  4=Medio alto  5=Alto
cod_grupo
1.0000     2492
2.0000     9775
3.0000    10157
4.0000     5024
5.0000     5043
NaN          28


---
## 6 · `crime_exposure_400m.parquet`
**Script 06** — `data/intermediate/crime_exposure_400m.parquet`

Índice de exposición al crimen para cada escuela, con buffer de **400 metros** y 
**6 ventanas temporales** previas al examen SIMCE (7, 14, 30, 90, 180, 365 días).

**Método IDW (Inverse Distance Weighting):**
```
peso_m     = 1 / (1 + distancia_metros)
indice_exp = sum(indice_m * peso_m * frec_m) / sum(peso_m * frec_m)
```
Resultado: 
- valor continuo \[0, 1\] por (RBD × año × grado × ventana_dias). 
-
El análisis principal del pre paper usa **ventana = 30 días**.

In [18]:
buf400 = pd.read_parquet(CRIME_BUFFER_400M_PATH)
info(buf400, 'crime_exposure_400m.parquet')
buf400.head(3)


------------------------------------------------------------
  crime_exposure_400m.parquet
------------------------------------------------------------
  Filas x Columnas : 595152 x 16
  Columnas         : ['rbd', 'agno', 'grado', 'ventana_dias', 'fecha_examen', 'indice_exp', 'frecuencia_total', 'n_manzanas', 'n_nivel_1', 'n_nivel_2', 'n_nivel_3', 'n_nivel_4', 'n_nivel_5', 'n_nivel_6', 'n_nivel_7', 'n_nivel_8']
  Memoria          : 135.1 MB
  Nulos            : ninguno



,rbd,agno,grado,ventana_dias,fecha_examen,indice_exp,frecuencia_total,n_manzanas,n_nivel_1,n_nivel_2,n_nivel_3,n_nivel_4,n_nivel_5,n_nivel_6,n_nivel_7,n_nivel_8
0,1646,2013,2b,7,2013-10-29,0.0000,0,0,0,0,0,0,0,0,0,0
1,6880,2013,2b,7,2013-10-29,0.0000,0,0,0,0,0,0,0,0,0,0
2,8485,2013,2b,7,2013-10-29,0.5771,75,20,0,7,16,23,1,10,7,11


In [19]:
print('Ventanas temporales disponibles (dias previos al examen):')
print(sorted(buf400['ventana_dias'].unique()))
print()
print('Indice de exposicion — estadisticas por ventana temporal:')
print(
    buf400.groupby('ventana_dias')['indice_exp']
    .describe()
    .round(4)
    .to_string()
)


Ventanas temporales disponibles (dias previos al examen):
[7, 14, 30, 90, 180, 365]

Indice de exposicion — estadisticas por ventana temporal:
                  count   mean    std    min    25%    50%    75%    max
ventana_dias                                                            
7            99192.0000 0.2446 0.2824 0.0000 0.0000 0.0000 0.5227 1.0000
14           99192.0000 0.2525 0.2769 0.0000 0.0000 0.0000 0.5283 0.9794
30           99192.0000 0.2539 0.2713 0.0000 0.0000 0.0000 0.5276 0.9485
90           99192.0000 0.2523 0.2666 0.0000 0.0000 0.0000 0.5228 0.9485
180          99192.0000 0.2512 0.2646 0.0000 0.0000 0.0000 0.5196 0.9485
365          99192.0000 0.2508 0.2636 0.0000 0.0000 0.0000 0.5185 0.9485


In [23]:
# Ejemplo: primer RBD disponible en todas sus ventanas
rbd_ej = int(buf400['rbd'].iloc[0])
print('Ejemplo RBD {}:'.format(rbd_ej))
print(
    buf400[buf400['rbd'] == rbd_ej]
    [['rbd','agno','grado','ventana_dias','indice_exp','frecuencia_total']]
    .sort_values(['agno','grado','ventana_dias'])
    .to_string(index=False)
)


Ejemplo RBD 1646:
 rbd  agno grado  ventana_dias  indice_exp  frecuencia_total
1646  2013    2b             7      0.0000                 0
1646  2013    2b            14      0.0000                 0
1646  2013    2b            30      0.0000                 0
1646  2013    2b            90      0.0000                 0
1646  2013    2b           180      0.0000                 0
1646  2013    2b           365      0.0000                 0
1646  2013    2m             7      0.0000                 0
1646  2013    2m            14      0.0000                 0
1646  2013    2m            30      0.0000                 0
1646  2013    2m            90      0.0000                 0
1646  2013    2m           180      0.0000                 0
1646  2013    2m           365      0.0000                 0
1646  2013    4b             7      0.0000                 0
1646  2013    4b            14      0.0000                 0
1646  2013    4b            30      0.0000                 0
1646  

---
## 7 · `tabla_final_escuelas_crimen.parquet`
**Script 07** — `data/processed/tabla_final_escuelas_crimen.parquet`

**Tabla maestra para análisis.** Une educación + crimen para todas las ventanas y radios. 
Es el único input del Script 08 (regresiones).

- Cada fila es RBD × año × grado × ventana_dias. 
- El flag `sin_cobertura_crimen` marca los ~661 RBDs en comunas periféricas sin datos 
policiales, los que se excluyen del análisis principal.

In [24]:
final = pd.read_parquet(FINAL_TABLE_PATH)
info(final, 'tabla_final_escuelas_crimen.parquet')
final.head(3)


------------------------------------------------------------
  tabla_final_escuelas_crimen.parquet
------------------------------------------------------------
  Filas x Columnas : 195114 x 57
  Columnas         : ['rbd', 'agno', 'grado', 'ventana_dias', 'nom_rbd', 'cod_com_rbd', 'nom_com_rbd', 'cod_reg_rbd', 'cod_depe2', 'cod_grupo', 'latitud', 'longitud', 'fecha_rendicion', 'prom_lect', 'prom_mate', 'nalu_lect', 'indice_exp_400m', 'frecuencia_total_400m', 'n_manzanas_400m', 'n_nivel_1_400m', 'n_nivel_2_400m', 'n_nivel_3_400m', 'n_nivel_4_400m', 'n_nivel_5_400m', 'n_nivel_6_400m', 'n_nivel_7_400m', 'n_nivel_8_400m', 'indice_exp_1km', 'frecuencia_total_1km', 'n_manzanas_1km', 'n_nivel_1_1km', 'n_nivel_2_1km', 'n_nivel_3_1km', 'n_nivel_4_1km', 'n_nivel_5_1km', 'n_nivel_6_1km', 'n_nivel_7_1km', 'n_nivel_8_1km', 'sin_cobertura_crimen', 'prom_lect2b_rbd', 'prom_lect4b_rbd', 'prom_mate4b_rbd', 'prom_lect6b_rbd', 'prom_mate6b_rbd', 'prom_lect8b_rbd', 'prom_mate8b_rbd', 'prom_lect2m_rbd', 'pr

,rbd,agno,grado,ventana_dias,nom_rbd,cod_com_rbd,nom_com_rbd,cod_reg_rbd,cod_depe2,cod_grupo,latitud,longitud,fecha_rendicion,prom_lect,prom_mate,nalu_lect,indice_exp_400m,frecuencia_total_400m,n_manzanas_400m,n_nivel_1_400m,...,n_nivel_8_1km,sin_cobertura_crimen,prom_lect2b_rbd,prom_lect4b_rbd,prom_mate4b_rbd,prom_lect6b_rbd,prom_mate6b_rbd,prom_lect8b_rbd,prom_mate8b_rbd,prom_lect2m_rbd,prom_mate2m_rbd,nalu_lect2b_rbd,nalu_lect4b_rbd,nalu_mate4b_rbd,nalu_lect6b_rbd,nalu_mate6b_rbd,nalu_lect8b_rbd,nalu_mate8b_rbd,nalu_lect2m_rbd,nalu_mate2m_rbd
0,8513,2013,2b,7,ESC. BAS. REPUBLICA ORIENTAL DE URUGUAY,13101.0000,SANTIAGO,13.0000,1.0000,3.0000,-33.4499,-70.6403,2013-10-29,247.0000,NaN,49.0000,0.5467,22,19,0,...,46,False,247.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8513,2013,2b,14,ESC. BAS. REPUBLICA ORIENTAL DE URUGUAY,13101.0000,SANTIAGO,13.0000,1.0000,3.0000,-33.4499,-70.6403,2013-10-29,247.0000,NaN,49.0000,0.5383,54,44,0,...,98,False,247.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8513,2013,2b,30,ESC. BAS. REPUBLICA ORIENTAL DE URUGUAY,13101.0000,SANTIAGO,13.0000,1.0000,3.0000,-33.4499,-70.6403,2013-10-29,247.0000,NaN,49.0000,0.5310,121,90,0,...,220,False,247.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
print('Dimensiones de la tabla:')
print('  Ventanas temporales :', sorted(final['ventana_dias'].unique()))
print('  Años                :', sorted(final['agno'].dropna().unique().tolist()))
print('  Grados              :', sorted(final['grado'].dropna().unique().tolist()))
print('  RBDs unicos (total) :', final['rbd'].nunique())


Dimensiones de la tabla:
  Ventanas temporales : [7, 14, 30, 90, 180, 365]
  Años                : [2013, 2014, 2015, 2016, 2017, 2018, 2019]
  Grados              : ['2b', '2m', '4b', '6b', '8b']
  RBDs unicos (total) : 1751


In [27]:
print('Cobertura geografica (ventana=30d):')
cob = final[final['ventana_dias'] == 30]['sin_cobertura_crimen'].value_counts()
total_30 = cob.sum()
for val, n in cob.items():
    label = 'Sin cobertura policial (excluidas)' if val else 'Con cobertura policial (analisis)'
    print('  {}: {:,} ({:.1f}%)'.format(label, n, 100*n/total_30))


Cobertura geografica (ventana=30d):
  Con cobertura policial (analisis): 27,614 (84.9%)
  Sin cobertura policial (excluidas): 4,905 (15.1%)


In [28]:
print('Comunas sin cobertura policial:')
sin_cob = (
    final[(final['sin_cobertura_crimen']) & (final['ventana_dias'] == 30)]
    .groupby(['cod_com_rbd', 'nom_com_rbd'])['rbd'].nunique()
    .reset_index().rename(columns={'rbd': 'n_rbds'})
    .sort_values('n_rbds', ascending=False)
)
print(sin_cob.to_string(index=False))


Comunas sin cobertura policial:
 cod_com_rbd       nom_com_rbd  n_rbds
      1.0000         MELIPILLA      44
  13501.0000         MELIPILLA      44
      1.0000            COLINA      39
  13301.0000            COLINA      39
     10.0000            COLINA      33
  13301.0000             Colin      33
  13301.0000             COLIN      31
  13601.0000         TALAGANTE      31
      2.0000          PEÑAFLOR      31
  13605.0000          PEÑAFLOR      31
      1.0000         TALAGANTE      31
  13605.0000             Peñaf      30
     50.0000          PEÑAFLOR      30
  13605.0000             PEÑAF      29
  13402.0000              BUIN      29
      2.0000              BUIN      29
  13601.0000             Talag      28
     52.0000         TALAGANTE      28
  13302.0000             LAMPA      28
      2.0000             LAMPA      28
     48.0000         MELIPILLA      27
      3.0000             PAINE      27
  13601.0000             TALAG      27
  13404.0000             PAINE  

---
## 8 · Universo de análisis limpio - comparativo con paper
Sanity check, se realizaron regresiones con el objetivo de comparar los resultados del paper con los outputs de este pipelline de datos.

Filtros aplicados para la especificación principal del paper:

```python
sin_cobertura_crimen == False   # excluye RBDs sin datos policiales
ventana_dias         == 30      # ventana principal del paper
radio                           # columnas indice_exp_400m
```

In [29]:
modelo = final[
    (~final['sin_cobertura_crimen']) &
    (final['ventana_dias'] == 30)
].copy()

print('Universo de analisis (ventana=30d, radio=400m, con cobertura policial):')
print('  Filas :', len(modelo))
print('  RBDs  :', modelo['rbd'].nunique())
print()
print('Distribucion por año x grado (N escuelas):')
print(modelo.groupby(['agno','grado'])['rbd'].nunique().unstack(fill_value=0).to_string())


Universo de analisis (ventana=30d, radio=400m, con cobertura policial):
  Filas : 27614
  RBDs  : 1417

Distribucion por año x grado (N escuelas):
grado    2b   2m    4b    6b    8b
agno                              
2013   1366  630  1365  1344  1289
2014   1380  641  1380  1361  1304
2015   1383  647  1384  1367  1306
2016      0  652   739     0     0
2017      0  658  1410     0  1322
2018      0  656  1374  1362     0
2019      0    0     0     0  1294


In [30]:
print('Indice de exposicion al crimen (400m, 30d) — estadisticas:')
print(modelo['indice_exp_400m'].describe().round(4).to_string())
print()
print('Puntajes SIMCE — estadisticas:')
score_cols = [c for c in modelo.columns if c.startswith('prom_')]
print(modelo[score_cols].describe().round(2).to_string())


Indice de exposicion al crimen (400m, 30d) — estadisticas:
count   27614.0000
mean        0.3999
std         0.2400
min         0.0000
25%         0.2900
50%         0.4952
75%         0.5689
max         0.9485

Puntajes SIMCE — estadisticas:
       prom_lect  prom_mate  prom_lect2b_rbd  prom_lect4b_rbd  prom_mate4b_rbd  prom_lect6b_rbd  prom_mate6b_rbd  prom_lect8b_rbd  prom_mate8b_rbd  prom_lect2m_rbd  prom_mate2m_rbd
count 27543.0000 23436.0000        4110.0000        7634.0000        7632.0000        5419.0000        5423.0000        6500.0000        6502.0000        3880.0000        3879.0000
mean    249.9800   258.5200         249.5800         261.0800         254.0900         242.3300         248.4000         238.9900         259.0900         257.6700         280.4700
std      29.2300    35.1900          25.8100          27.0100          29.3200          28.0100          31.9300          28.1400          31.2100          30.2700          45.5000
min       0.0000     0.0000      

In [31]:
print('Correlacion bruta entre indice de exposicion (400m, 30d) y puntajes SIMCE:')
print('(sin controles — solo descriptivo)')
score_cols = [c for c in modelo.columns if c.startswith('prom_')]
corr = modelo[['indice_exp_400m'] + score_cols].corr()['indice_exp_400m'].drop('indice_exp_400m')
print(corr.round(4).to_string())


Correlacion bruta entre indice de exposicion (400m, 30d) y puntajes SIMCE:
(sin controles — solo descriptivo)
prom_lect         -0.0573
prom_mate         -0.0705
prom_lect2b_rbd   -0.0469
prom_lect4b_rbd   -0.0640
prom_mate4b_rbd   -0.0689
prom_lect6b_rbd   -0.0604
prom_mate6b_rbd   -0.0705
prom_lect8b_rbd   -0.0651
prom_mate8b_rbd   -0.0819
prom_lect2m_rbd   -0.0345
prom_mate2m_rbd   -0.0273


----------------
----------------